<a href="https://colab.research.google.com/github/dinujakr/Statistical-Learning-e20190/blob/main/data_wrangling_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Background
In real-world data science, 80% of the work is spent cleaning and exploring data. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to build a **Reusable Python Class** named `DataInspector` and a supporting `PlottingMethods` class that can be imported into Google Colab to automate these tasks.

### The Objective
Develop an end-to-end tool for CSV data ingestion, advanced cleaning, feature engineering preparation, and high-level statistical visualization.

### Technical Requirements

#### 1. Data Ingestion & Sanitization
* **Colab Integration**: Implement `upload_data()` to handle local file uploads.
* **Garbage String Handling**: Automatically recognize and convert strings like `'?'`, `'n/a'`, `'NULL'`, and `' '` into actual `NaN` values.
* **Auto-Type Correction**: Force-convert columns to numeric types if the conversion does not result in an entirely null column.

#### 2. Structural Analysis & Cleaning
* **Data Summary**: Provide a method to display row/column counts, a preview of the first 20 rows, and a breakdown of numerical vs. categorical columns.
* **Intelligent Imputation**: Create a `handle_missing_values()` method supporting multiple strategies: `mean`, `median`, `mode`, or a `constant` value.
* **Duplicate & Outlier Management**:
    * Implement `remove_duplicates()` to prune exact row matches.
    * Develop an **IQR-based** outlier detection system (`handle_outliers`) that allows users to flag or automatically delete rows based on specific columns.
* **Targeted Deletion**: Implement interactive methods (`delete_rows`, `delete_columns`) that accept comma-separated user input to prune the dataset.

#### 3. Feature Engineering Preparation (Normalization)
* **Numeric Scaling**: Implement `extract_normalized_numeric_data()` supporting `minmax`, `standard` (Z-score), and `robust` (IQR-based) scaling.
* **Categorical Encoding**: Implement `extract_normalized_categorical_data()` supporting `onehot`, `ordinal`, and `uniform` (scaled 0-1) encoding.
* **Dataset Merging**: Provide a method to create a unified DataFrame containing original numeric data alongside encoded categorical data.

#### 4. Advanced Interactive Visualization (Plotly)
* **Univariate Subplots**: For numeric columns, generate a 3-panel subplot: **Horizontal Violin/Box**, **Scatter Plot** (Index vs Value), and **Histogram**.
* **Smart Relationships**: Build a `plot_relationship()` tool that detects types and chooses the correct chart:
    * **Num-Num**: Scatter with OLS Trendline.
    * **Cat-Num**: Box plot with all data points.
    * **Cat-Cat**: Grouped Bar chart.
* **Categorical Frequency**: Create bar charts displaying both raw counts and percentage labels.

#### 5. Deep Statistical Insights
* **Unified Heatmap**: Develop `plot_all_associations_heatmap()` to visualize relationships across *all* data types:
    * **Numeric-Numeric**: Pearson's $r$.
    * **Categorical-Categorical**: Cram\u00e9r's $V$.
    * **Mixed (Num-Cat)**: Point-Biserial correlation or Eta (via ANOVA).

#### 6. Custom Modular Plotting
Implement a separate `PlottingMethods` class to handle granular chart generation (Bar, Pie, Histogram) that returns HTML-wrapped figures for flexible embedding.

### Submission Criteria
1.  **Object-Oriented Design**: All logic must be encapsulated within the `DataInspector` and `PlottingMethods` classes.
2.  **Clean Code**: Every method must include descriptive **Docstrings** and handle empty/None data gracefully.
3.  **Real-world Testing**: Demonstrate the tool using a dataset (e.g., Titanic) by performing a full flow: Upload $\rightarrow$ Impute $\rightarrow$ Normalize $\rightarrow$ Visualize Associations.

---
## Implementation
---

### Step 0 : Import Libraries

In [1]:
import io
import warnings
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

from scipy.stats import chi2_contingency, pearsonr
from sklearn.preprocessing import OrdinalEncoder
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
print('All imports successful.')

All imports successful.


---
### Sections : `DataInspector` Class

| Section | Capability | Key Methods |
|---------|-----------|-------------|
| 1 | Data Ingestion & Sanitization | `upload_data`, `load_from_dataframe`, `_sanitize` |
| 2 | Structural Analysis & Cleaning | `data_summary`, `handle_missing_values`, `remove_duplicates`, `handle_outliers`, `delete_rows`, `delete_columns` |
| 3 | Feature Engineering | `extract_normalized_numeric_data`, `extract_normalized_categorical_data`, `merge_normalized_data` |
| 4 | Advanced Visualization | `plot_univariate`, `plot_relationship`, `plot_categorical_frequency` |
| 5 | Deep Statistical Insights | `plot_all_associations_heatmap`, `_cramers_v`, `_correlation_ratio` |

In [2]:
class DataInspector:
    """
    End-to-end tool for CSV data ingestion, advanced cleaning,
    feature engineering preparation, and high-level statistical
    visualisation.

    Attributes
    ----------
    df               : pd.DataFrame or None \u2013 the working dataset.
    numeric_cols     : list[str] \u2013 column names with numeric dtypes.
    categorical_cols : list[str] \u2013 column names with object/category dtypes.
    """

    # Strings treated as missing
    _GARBAGE_STRINGS = [
        '?', 'n/a', 'N/A', 'na', 'NA', 'NULL', 'null',
        'None', 'none', '--', '-', ' ', '', 'nan', 'NaN'
    ]

    # ================================================================
    #  INITIALISATION
    # ================================================================
    def __init__(self):
        """Initialise an empty DataInspector instance."""
        self.df = None
        self.numeric_cols = []
        self.categorical_cols = []

    # ================================================================
    #  SECTION 1 : DATA INGESTION & SANITIZATION
    # ================================================================
    def upload_data(self):
        """Upload a CSV via Google Colab file-upload dialog.

        Falls back to a manual path prompt when running outside Colab.
        The loaded data is automatically sanitised.
        """
        try:
            from google.colab import files
            uploaded = files.upload()
            filename = list(uploaded.keys())[0]
            self.df = pd.read_csv(
                io.BytesIO(uploaded[filename]),
                na_values=self._GARBAGE_STRINGS
            )
            print(f'Uploaded: {filename}')
        except ImportError:
            path = input('Colab not detected. Enter CSV file path: ')
            self.df = pd.read_csv(path, na_values=self._GARBAGE_STRINGS)
            print(f'Loaded from path: {path}')

        self._sanitize()
        print(f'Shape: {self.df.shape[0]} rows x {self.df.shape[1]} columns')

    def load_from_dataframe(self, df):
        """Load data from an existing DataFrame.

        Parameters
        ----------
        df : pd.DataFrame \u2013 a copy is stored internally.
        """
        if df is None or not isinstance(df, pd.DataFrame):
            print('Please provide a valid pandas DataFrame.')
            return
        self.df = df.copy()
        self._sanitize()
        print(f'Data loaded from DataFrame.')
        print(f'Shape: {self.df.shape[0]} rows x {self.df.shape[1]} columns')

    def _sanitize(self):
        """Replace garbage strings with NaN, strip whitespace,
        and auto-convert object columns to numeric where possible."""
        if self.df is None:
            return
        # Replace known garbage strings
        self.df.replace(self._GARBAGE_STRINGS, np.nan, inplace=True)
        # Strip whitespace in object columns
        for col in self.df.select_dtypes(include='object').columns:
            self.df[col] = self.df[col].astype(str).str.strip()
            self.df[col] = self.df[col].replace(
                self._GARBAGE_STRINGS + [''], np.nan
            )
        # Auto-type correction
        for col in list(self.df.select_dtypes(include='object').columns):
            converted = pd.to_numeric(self.df[col], errors='coerce')
            if not converted.isna().all():
                self.df[col] = converted
        self._refresh_column_types()

    def _refresh_column_types(self):
        """Recalculate numeric and categorical column lists."""
        self.numeric_cols = (
            self.df.select_dtypes(include=np.number).columns.tolist()
        )
        self.categorical_cols = (
            self.df.select_dtypes(include=['object', 'category']).columns.tolist()
        )

    # ================================================================
    #  SECTION 2 : STRUCTURAL ANALYSIS & CLEANING
    # ================================================================
    def data_summary(self):
        """Print row/column counts, first 20 rows,
        numeric vs categorical breakdown, and missing-value report."""
        if self.df is None:
            print('No data loaded. Call upload_data() first.')
            return
        print('=' * 60)
        print('DATASET SUMMARY')
        print('=' * 60)
        print(f'Rows   : {self.df.shape[0]}')
        print(f'Columns: {self.df.shape[1]}')
        print('\nFirst 20 Rows:')
        display(self.df.head(20))
        print(f'\nNumeric Columns ({len(self.numeric_cols)}):')
        print(f"  {', '.join(self.numeric_cols) if self.numeric_cols else '(none)'}")
        print(f'\nCategorical Columns ({len(self.categorical_cols)}):')
        print(f"  {', '.join(self.categorical_cols) if self.categorical_cols else '(none)'}")
        print('\nMissing Values:')
        missing = self.df.isnull().sum()
        missing = missing[missing > 0]
        if len(missing) > 0:
            for col_name, count in missing.items():
                pct = count / len(self.df) * 100
                print(f'  {col_name}: {count} ({pct:.1f}%)')
        else:
            print('  None - dataset is complete!')
        print('\nData Types:')
        print(self.df.dtypes.value_counts().to_string())
        print('=' * 60)

    def handle_missing_values(self, strategy='mean', columns=None,
                              fill_value=None):
        """Impute missing values.

        Parameters
        ----------
        strategy   : {'mean', 'median', 'mode', 'constant'}
        columns    : str, list[str], or None
        fill_value : scalar (required when strategy='constant')
        """
        if self.df is None:
            print('No data loaded.')
            return
        if columns is None:
            columns = (
                self.numeric_cols if strategy in ('mean', 'median')
                else self.df.columns.tolist()
            )
        if isinstance(columns, str):
            columns = [columns]
        filled_total = 0
        for col in columns:
            if col not in self.df.columns:
                print(f"Column '{col}' not found - skipping.")
                continue
            nulls_before = self.df[col].isnull().sum()
            if nulls_before == 0:
                continue
            if strategy == 'mean':
                self.df[col] = self.df[col].fillna(self.df[col].mean())
            elif strategy == 'median':
                self.df[col] = self.df[col].fillna(self.df[col].median())
            elif strategy == 'mode':
                mode_val = self.df[col].mode()
                if len(mode_val) > 0:
                    self.df[col] = self.df[col].fillna(mode_val[0])
            elif strategy == 'constant':
                if fill_value is None:
                    print("fill_value is required for 'constant' strategy.")
                    return
                self.df[col] = self.df[col].fillna(fill_value)
            else:
                print(f"Unknown strategy '{strategy}'.")
                return
            filled = nulls_before - self.df[col].isnull().sum()
            filled_total += filled
            print(f"  '{col}': filled {filled} nulls using {strategy}")
        print(f'Total values imputed: {filled_total}')
        self._refresh_column_types()

    def remove_duplicates(self):
        """Remove exact duplicate rows."""
        if self.df is None:
            print('No data loaded.')
            return
        before = len(self.df)
        self.df.drop_duplicates(inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        removed = before - len(self.df)
        print(f'Removed {removed} duplicate row(s). '
              f'Remaining: {len(self.df)} rows.')

    def handle_outliers(self, columns=None, action='flag'):
        """IQR-based outlier detection.

        Parameters
        ----------
        columns : str, list[str], or None (defaults to all numeric)
        action  : 'flag' adds a boolean column; 'delete' removes rows.
        """
        if self.df is None:
            print('No data loaded.')
            return
        if columns is None:
            columns = self.numeric_cols
        if isinstance(columns, str):
            columns = [columns]
        total_outliers = 0
        for col in columns:
            if col not in self.numeric_cols:
                print(f"'{col}' is not numeric - skipping.")
                continue
            Q1    = self.df[col].quantile(0.25)
            Q3    = self.df[col].quantile(0.75)
            IQR   = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mask  = (self.df[col] < lower) | (self.df[col] > upper)
            n_out = mask.sum()
            total_outliers += n_out
            print(f"  '{col}': {n_out} outlier(s) "
                  f'[valid range: {lower:.2f} to {upper:.2f}]')
            if action == 'flag':
                self.df[f'{col}_outlier'] = mask
            elif action == 'delete':
                self.df = self.df[~mask]
        if action == 'delete':
            self.df.reset_index(drop=True, inplace=True)
            self._refresh_column_types()
            print(f'Deleted outlier rows. Remaining: {len(self.df)}.')
        elif action == 'flag':
            print(f'Flagged {total_outliers} outlier instance(s).')

    def delete_rows(self, indices=None):
        """Delete rows by index (comma-separated string or list)."""
        if self.df is None:
            print('No data loaded.')
            return
        if indices is None:
            indices = input('Enter row indices to delete (comma-separated): ')
        if isinstance(indices, str):
            indices = [int(i.strip()) for i in indices.split(',') if i.strip()]
        before = len(self.df)
        self.df.drop(index=indices, inplace=True, errors='ignore')
        self.df.reset_index(drop=True, inplace=True)
        print(f'Deleted {before - len(self.df)} row(s). '
              f'Remaining: {len(self.df)} rows.')

    def delete_columns(self, columns=None):
        """Delete columns by name (comma-separated string or list)."""
        if self.df is None:
            print('No data loaded.')
            return
        if columns is None:
            columns = input('Enter column names to delete (comma-separated): ')
        if isinstance(columns, str):
            columns = [c.strip() for c in columns.split(',') if c.strip()]
        before = len(self.df.columns)
        self.df.drop(columns=columns, inplace=True, errors='ignore')
        self._refresh_column_types()
        print(f'Deleted {before - len(self.df.columns)} column(s). '
              f'Remaining: {len(self.df.columns)} columns.')

    # ================================================================
    #  SECTION 3 : FEATURE ENGINEERING PREPARATION
    # ================================================================
    def extract_normalized_numeric_data(self, method='minmax'):
        """Scale numeric columns.

        Parameters
        ----------
        method : {'minmax', 'standard', 'robust'}

        Returns
        -------
        pd.DataFrame
        """
        if self.df is None or len(self.numeric_cols) == 0:
            print('No numeric data available.')
            return pd.DataFrame()
        data = self.df[self.numeric_cols].copy()
        if method == 'minmax':
            denom  = (data.max() - data.min()).replace(0, 1)
            result = (data - data.min()) / denom
        elif method == 'standard':
            std    = data.std().replace(0, 1)
            result = (data - data.mean()) / std
        elif method == 'robust':
            Q1  = data.quantile(0.25)
            Q3  = data.quantile(0.75)
            IQR = (Q3 - Q1).replace(0, 1)
            result = (data - data.median()) / IQR
        else:
            print(f"Unknown method '{method}'. Use 'minmax','standard','robust'.")
            return pd.DataFrame()
        print(f"Numeric data normalised ('{method}'). Shape: {result.shape}")
        return result

    def extract_normalized_categorical_data(self, method='onehot'):
        """Encode categorical columns.

        Parameters
        ----------
        method : {'onehot', 'ordinal', 'uniform'}

        Returns
        -------
        pd.DataFrame
        """
        if self.df is None or len(self.categorical_cols) == 0:
            print('No categorical data available.')
            return pd.DataFrame()
        data = self.df[self.categorical_cols].copy()
        if method == 'onehot':
            result = pd.get_dummies(data, dtype=int)
        elif method == 'ordinal':
            enc = OrdinalEncoder(handle_unknown='use_encoded_value',
                                unknown_value=-1)
            encoded = enc.fit_transform(data.fillna('__MISSING__'))
            result  = pd.DataFrame(encoded, columns=data.columns,
                                   index=data.index)
        elif method == 'uniform':
            enc = OrdinalEncoder(handle_unknown='use_encoded_value',
                                unknown_value=-1)
            encoded = enc.fit_transform(data.fillna('__MISSING__'))
            result  = pd.DataFrame(encoded, columns=data.columns,
                                   index=data.index)
            for c in result.columns:
                c_min, c_max = result[c].min(), result[c].max()
                if c_max > c_min:
                    result[c] = (result[c] - c_min) / (c_max - c_min)
                else:
                    result[c] = 0.0
        else:
            print(f"Unknown method '{method}'. Use 'onehot','ordinal','uniform'.")
            return pd.DataFrame()
        print(f"Categorical data encoded ('{method}'). Shape: {result.shape}")
        return result

    def merge_normalized_data(self, numeric_method='minmax',
                              categorical_method='onehot'):
        """Unified DataFrame of scaled numeric + encoded categorical data.

        Returns
        -------
        pd.DataFrame
        """
        num_data = self.extract_normalized_numeric_data(method=numeric_method)
        cat_data = self.extract_normalized_categorical_data(
            method=categorical_method)
        merged = pd.concat([num_data, cat_data], axis=1)
        print(f'Merged dataset shape: {merged.shape}')
        return merged

    # ================================================================
    #  SECTION 4 : ADVANCED INTERACTIVE VISUALISATION
    # ================================================================
    def plot_univariate(self, col):
        """3-panel subplot: Horizontal Violin/Box | Scatter | Histogram.

        Parameters
        ----------
        col : str \u2013 numeric column name
        """
        if self.df is None:
            print('No data loaded.')
            return
        if col not in self.numeric_cols:
            print(f"'{col}' is not a numeric column.")
            return
        data = self.df[col].dropna()
        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=['Violin / Box Plot',
                            'Scatter (Index vs Value)',
                            'Histogram'],
            column_widths=[0.33, 0.33, 0.34]
        )
        # Panel 1 - Horizontal Violin + Box
        fig.add_trace(go.Violin(
            x=data, box_visible=True, meanline_visible=True,
            orientation='h', name='Distribution',
            fillcolor='rgba(99,110,250,0.5)',
            line_color='rgb(99,110,250)'
        ), row=1, col=1)
        # Panel 2 - Scatter
        fig.add_trace(go.Scatter(
            x=list(range(len(data))), y=data.values,
            mode='markers', name='Values',
            marker=dict(size=4, color='rgb(239,85,59)', opacity=0.6)
        ), row=1, col=2)
        # Panel 3 - Histogram
        fig.add_trace(go.Histogram(
            x=data, name='Frequency',
            marker_color='rgb(0,204,150)', opacity=0.75
        ), row=1, col=3)
        fig.update_layout(
            title=f'Univariate Analysis: {col}',
            showlegend=False, height=450,
            template='plotly_white'
        )
        fig.show()

    def plot_relationship(self, col1, col2):
        """Smart relationship plot (auto-detects column types).

        * Num-Num  : Scatter + OLS trendline
        * Cat-Num  : Box plot with all data points
        * Cat-Cat  : Grouped bar chart

        Parameters
        ----------
        col1, col2 : str
        """
        if self.df is None:
            print('No data loaded.')
            return
        for c in (col1, col2):
            if c not in self.df.columns:
                print(f"Column '{c}' not found.")
                return
        c1_num = col1 in self.numeric_cols
        c2_num = col2 in self.numeric_cols
        if c1_num and c2_num:
            fig = px.scatter(self.df, x=col1, y=col2,
                             trendline='ols',
                             title=f'{col1} vs {col2} (OLS Trendline)',
                             template='plotly_white')
            fig.show()
        elif c1_num != c2_num:
            cat_col = col1 if not c1_num else col2
            num_col = col2 if not c1_num else col1
            fig = px.box(self.df, x=cat_col, y=num_col,
                         points='all',
                         title=f'{num_col} by {cat_col}',
                         template='plotly_white')
            fig.show()
        else:
            ct = pd.crosstab(self.df[col1], self.df[col2])
            fig = px.bar(ct, barmode='group',
                         title=f'{col1} vs {col2} (Grouped Bar)',
                         template='plotly_white')
            fig.update_layout(xaxis_title=col1, yaxis_title='Count')
            fig.show()

    def plot_categorical_frequency(self, col):
        """Bar chart with raw counts and percentage labels.

        Parameters
        ----------
        col : str \u2013 categorical column name
        """
        if self.df is None:
            print('No data loaded.')
            return
        if col not in self.categorical_cols:
            print(f"'{col}' is not a categorical column.")
            return
        counts = self.df[col].value_counts()
        total  = counts.sum()
        pcts   = (counts / total * 100).round(1)
        text   = [f'{c} ({p}%)' for c, p in zip(counts.values, pcts.values)]
        fig = go.Figure(go.Bar(
            x=counts.index.astype(str), y=counts.values,
            text=text, textposition='outside',
            marker_color='rgb(99,110,250)'
        ))
        fig.update_layout(
            title=f'Frequency: {col}',
            xaxis_title=col, yaxis_title='Count',
            template='plotly_white', height=450
        )
        fig.show()

    # ================================================================
    #  SECTION 5 : DEEP STATISTICAL INSIGHTS
    # ================================================================
    def _cramers_v(self, x, y):
        """Cramer's V between two categorical Series."""
        confusion = pd.crosstab(x, y)
        chi2    = chi2_contingency(confusion)[0]
        n       = confusion.sum().sum()
        min_dim = min(confusion.shape) - 1
        if min_dim == 0 or n == 0:
            return 0.0
        return np.sqrt(chi2 / (n * min_dim))

    def _correlation_ratio(self, cat, num):
        """Eta (correlation ratio) via one-way ANOVA decomposition."""
        tmp = pd.DataFrame({'cat': cat, 'num': num}).dropna()
        if len(tmp) == 0:
            return 0.0
        categories = tmp['cat'].unique()
        if len(categories) <= 1:
            return 0.0
        grand_mean = tmp['num'].mean()
        ss_between = sum(
            len(tmp[tmp['cat'] == c])
            * (tmp[tmp['cat'] == c]['num'].mean() - grand_mean) ** 2
            for c in categories
        )
        ss_total = ((tmp['num'] - grand_mean) ** 2).sum()
        if ss_total == 0:
            return 0.0
        return np.sqrt(ss_between / ss_total)

    def plot_all_associations_heatmap(self):
        """Unified N x N association heatmap.

        * Numeric-Numeric       : Pearson's r
        * Categorical-Categorical: Cramer's V
        * Mixed                 : Correlation Ratio (Eta)
        """
        if self.df is None:
            print('No data loaded.')
            return
        cols = self.numeric_cols + self.categorical_cols
        n = len(cols)
        if n == 0:
            print('No columns to analyse.')
            return
        matrix = pd.DataFrame(np.zeros((n, n)),
                              index=cols, columns=cols)
        for i in range(n):
            for j in range(n):
                if i == j:
                    matrix.iloc[i, j] = 1.0
                    continue
                if j < i:
                    matrix.iloc[i, j] = matrix.iloc[j, i]
                    continue
                ci, cj = cols[i], cols[j]
                i_num  = ci in self.numeric_cols
                j_num  = cj in self.numeric_cols
                try:
                    if i_num and j_num:
                        valid = self.df[[ci, cj]].dropna()
                        if len(valid) > 2:
                            corr, _ = pearsonr(valid[ci], valid[cj])
                            matrix.iloc[i, j] = corr
                    elif (not i_num) and (not j_num):
                        valid = self.df[[ci, cj]].dropna()
                        matrix.iloc[i, j] = self._cramers_v(
                            valid[ci], valid[cj])
                    else:
                        cat_c = ci if not i_num else cj
                        num_c = cj if not i_num else ci
                        matrix.iloc[i, j] = self._correlation_ratio(
                            self.df[cat_c], self.df[num_c])
                except Exception:
                    matrix.iloc[i, j] = 0.0
        fig = go.Figure(data=go.Heatmap(
            z=matrix.values,
            x=matrix.columns.tolist(),
            y=matrix.index.tolist(),
            colorscale='RdBu_r', zmin=-1, zmax=1,
            text=np.round(matrix.values, 2).astype(str),
            texttemplate='%{text}',
            textfont=dict(size=10),
            hovertemplate='%{y} vs %{x}: %{z:.3f}<extra></extra>'
        ))
        fig.update_layout(
            title="Unified Association Heatmap (Pearson | Cramer's V | Eta)",
            height=max(500, 100 + n * 40),
            width=max(600, 100 + n * 40),
            template='plotly_white',
            xaxis=dict(tickangle=45)
        )
        fig.show()


print('DataInspector class defined.')

DataInspector class defined.


---
## Real-World Testing : Titanic Dataset

Full pipeline: **Upload \u2192 Impute \u2192 Normalize \u2192 Visualize Associations**

---

In [3]:
TITANIC_URL = (
    'https://raw.githubusercontent.com/datasciencedojo/datasets'
    '/master/titanic.csv'
)
raw_df = pd.read_csv(TITANIC_URL)
print(f'Downloaded Titanic dataset: {raw_df.shape}')

Downloaded Titanic dataset: (891, 12)


In [4]:
inspector = DataInspector()
inspector.load_from_dataframe(raw_df)

Data loaded from DataFrame.
Shape: 891 rows x 12 columns


### Step 2 : Data Summary  *(Section 2)*

In [5]:
inspector.data_summary()

DATASET SUMMARY
Rows   : 891
Columns: 12

First 20 Rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877.0,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463.0,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909.0,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742.0,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736.0,30.0708,NaN,C



Numeric Columns (8):
  PassengerId, Survived, Pclass, Age, SibSp, Parch, Ticket, Fare

Categorical Columns (4):
  Name, Sex, Cabin, Embarked

Missing Values:
  Age: 177 (19.9%)
  Ticket: 230 (25.8%)
  Cabin: 687 (77.1%)
  Embarked: 2 (0.2%)

Data Types:
int64      5
object     4
float64    3


### Step 3 : Handle Missing Values  *(Section 2)*

In [6]:
# Impute Age with median (robust to outliers / skew)
inspector.handle_missing_values(strategy='median', columns=['Age'])

  'Age': filled 177 nulls using median
Total values imputed: 177


In [7]:
# Impute Embarked with mode (most frequent port)
inspector.handle_missing_values(strategy='mode', columns=['Embarked'])

  'Embarked': filled 2 nulls using mode
Total values imputed: 2


In [8]:
# Cabin has 77% nulls - drop the entire column
inspector.delete_columns('Cabin')

Deleted 1 column(s). Remaining: 11 columns.


In [9]:
# Verify remaining nulls
remaining = inspector.df.isnull().sum()
remaining = remaining[remaining > 0]
print('Remaining nulls:')
print(remaining if len(remaining) > 0 else 'None - dataset is clean!')

Remaining nulls:
Ticket    230
dtype: int64


### Step 4 : Remove Duplicates  *(Section 2)*

In [10]:
inspector.remove_duplicates()

Removed 0 duplicate row(s). Remaining: 891 rows.


### Step 5 : Handle Outliers  *(Section 2)*

In [11]:
# Flag outliers in Fare using IQR method
inspector.handle_outliers(columns=['Fare'], action='flag')

  'Fare': 116 outlier(s) [valid range: -26.72 to 65.63]
Flagged 116 outlier instance(s).


In [12]:
# Check the outlier distribution
print('Fare outlier distribution:')
print(inspector.df['Fare_outlier'].value_counts())

Fare outlier distribution:
Fare_outlier
False    775
True     116
Name: count, dtype: int64


### Step 6 : Targeted Deletion  *(Section 2)*

In [13]:
# Drop the outlier flag and non-useful identifier columns
inspector.delete_columns('Fare_outlier, Name, Ticket, PassengerId')

Deleted 4 column(s). Remaining: 8 columns.


In [14]:
# Current state
print(f'Columns: {list(inspector.df.columns)}')
print(f'Shape  : {inspector.df.shape}')

Columns: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
Shape  : (891, 8)


### Step 7 : Numeric Normalization  *(Section 3)*

In [15]:
# Min-Max scaling
num_minmax = inspector.extract_normalized_numeric_data(method='minmax')
display(num_minmax.head())

Numeric data normalised ('minmax'). Shape: (891, 6)


,Survived,Pclass,Age,SibSp,Parch,Fare
0,0.0,1.0,0.271174,0.125,0.0,0.014151
1,1.0,0.0,0.472229,0.125,0.0,0.139136
2,1.0,1.0,0.321438,0.000,0.0,0.015469
3,1.0,0.0,0.434531,0.125,0.0,0.103644
4,0.0,1.0,0.434531,0.000,0.0,0.015713


In [16]:
# Standard (Z-score) scaling
num_standard = inspector.extract_normalized_numeric_data(method='standard')
display(num_standard.head())

Numeric data normalised ('standard'). Shape: (891, 6)


,Survived,Pclass,Age,SibSp,Parch,Fare
0,-0.788829,0.826913,-0.565419,0.432550,-0.473408,-0.502163
1,1.266279,-1.565228,0.663488,0.432550,-0.473408,0.786404
2,1.266279,0.826913,-0.258192,-0.474279,-0.473408,-0.488580
3,1.266279,-1.565228,0.433068,0.432550,-0.473408,0.420494
4,-0.788829,0.826913,0.433068,-0.474279,-0.473408,-0.486064


In [17]:
# Robust (IQR-based) scaling
num_robust = inspector.extract_normalized_numeric_data(method='robust')
display(num_robust.head())

Numeric data normalised ('robust'). Shape: (891, 6)


,Survived,Pclass,Age,SibSp,Parch,Fare
0,0.0,0.0,-0.461538,1.0,0.0,-0.312011
1,1.0,-2.0,0.769231,1.0,0.0,2.461242
2,1.0,0.0,-0.153846,0.0,0.0,-0.282777
3,1.0,-2.0,0.538462,1.0,0.0,1.673732
4,0.0,0.0,0.538462,0.0,0.0,-0.277363


### Step 8 : Categorical Encoding  *(Section 3)*

In [18]:
# One-Hot encoding
cat_onehot = inspector.extract_normalized_categorical_data(method='onehot')
display(cat_onehot.head())

Categorical data encoded ('onehot'). Shape: (891, 5)


,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0,1,0,0,1
1,1,0,1,0,0
2,1,0,0,0,1
3,1,0,0,0,1
4,0,1,0,0,1


In [ ]:
# Ordinal encoding
cat_ordinal = inspector.extract_normalized_categorical_data(method='ordinal')
display(cat_ordinal.head())

In [ ]:
# Uniform encoding (ordinal scaled to [0, 1])
cat_uniform = inspector.extract_normalized_categorical_data(method='uniform')
display(cat_uniform.head())

### Step 9 : Merge Normalized Dataset  *(Section 3)*

In [ ]:
merged = inspector.merge_normalized_data(
    numeric_method='standard',
    categorical_method='onehot'
)
display(merged.head())

### Step 10 : Univariate Visualization  *(Section 4)*

3-panel subplot per column: **Horizontal Violin/Box** | **Scatter** | **Histogram**

In [19]:
inspector.plot_univariate('Age')

In [ ]:
inspector.plot_univariate('Fare')

### Step 11 : Smart Relationship Plots  *(Section 4)*

Auto-detection: **Num-Num** \u2192 Scatter + OLS | **Cat-Num** \u2192 Box | **Cat-Cat** \u2192 Grouped Bar

In [ ]:
# Numeric vs Numeric -> Scatter + OLS trendline
inspector.plot_relationship('Age', 'Fare')

In [ ]:
# Categorical vs Numeric -> Box plot with all data points
inspector.plot_relationship('Sex', 'Fare')

In [ ]:
# Categorical vs Categorical -> Grouped bar chart
inspector.plot_relationship('Sex', 'Embarked')

### Step 12 : Categorical Frequency  *(Section 4)*

In [ ]:
inspector.plot_categorical_frequency('Embarked')

In [ ]:
inspector.plot_categorical_frequency('Sex')

### Step 13 : Unified Association Heatmap  *(Section 5)*

Single heatmap combining Pearson's r, Cramer's V, and Eta across all columns.

In [20]:
inspector.plot_all_associations_heatmap()

### Step 14 : PlottingMethods Demo  *(Section 6)*

Each method returns an **HTML string** that can be embedded with `display(HTML(...))`.

---
### Section 6 : `PlottingMethods` Class \u2014 Custom Modular Plotting

A standalone utility class for granular chart generation.  
Every method returns an **HTML string** (via `plotly.io.to_html`) so the
figure can be embedded anywhere with `display(HTML(...))`.

In [21]:
class PlottingMethods:
    """
    Modular plotting class for granular chart generation.

    Each static method accepts a DataFrame and a column name,
    builds a Plotly figure, and returns the figure wrapped as an
    HTML string suitable for embedding via IPython.display.HTML.
    """

    # ---- helper ----
    @staticmethod
    def to_html(fig):
        """Convert a Plotly figure to an embeddable HTML string.

        Parameters
        ----------
        fig : plotly.graph_objects.Figure or None

        Returns
        -------
        str or None
        """
        if fig is None:
            return None
        return pio.to_html(fig, full_html=False, include_plotlyjs='cdn')

    # ---- Bar Chart ----
    @staticmethod
    def bar_chart(df, col, title=None):
        """Vertical bar chart of value counts with count and percentage labels.

        Parameters
        ----------
        df    : pd.DataFrame
        col   : str \u2013 column name
        title : str, optional

        Returns
        -------
        str or None \u2013 HTML string of the figure
        """
        if df is None or col not in df.columns:
            print(f"Invalid DataFrame or column '{col}'.")
            return None
        counts = df[col].value_counts()
        total  = counts.sum()
        pcts   = (counts / total * 100).round(1)
        text   = [f'{c} ({p}%)' for c, p in zip(counts.values, pcts.values)]
        fig = go.Figure(go.Bar(
            x=counts.index.astype(str), y=counts.values,
            text=text, textposition='outside',
            marker_color='rgb(99,110,250)'
        ))
        fig.update_layout(title=title or f'Bar Chart: {col}',
                          xaxis_title=col, yaxis_title='Count',
                          template='plotly_white', height=450)
        return PlottingMethods.to_html(fig)

    # ---- Pie / Donut Chart ----
    @staticmethod
    def pie_chart(df, col, title=None):
        """Donut-style pie chart of categorical distribution.

        Parameters
        ----------
        df    : pd.DataFrame
        col   : str
        title : str, optional

        Returns
        -------
        str or None \u2013 HTML string
        """
        if df is None or col not in df.columns:
            print(f"Invalid DataFrame or column '{col}'.")
            return None
        counts = df[col].value_counts()
        fig = go.Figure(go.Pie(
            labels=counts.index.astype(str), values=counts.values,
            hole=0.4, textinfo='label+percent',
            marker=dict(line=dict(color='white', width=2))
        ))
        fig.update_layout(title=title or f'Distribution: {col}',
                          template='plotly_white', height=450)
        return PlottingMethods.to_html(fig)

    # ---- Histogram ----
    @staticmethod
    def histogram(df, col, bins=30, title=None):
        """Histogram for a numeric column.

        Parameters
        ----------
        df    : pd.DataFrame
        col   : str
        bins  : int (default 30)
        title : str, optional

        Returns
        -------
        str or None \u2013 HTML string
        """
        if df is None or col not in df.columns:
            print(f"Invalid DataFrame or column '{col}'.")
            return None
        fig = go.Figure(go.Histogram(
            x=df[col].dropna(), nbinsx=bins,
            marker_color='rgb(0,204,150)', opacity=0.75
        ))
        fig.update_layout(title=title or f'Histogram: {col}',
                          xaxis_title=col, yaxis_title='Frequency',
                          template='plotly_white', height=450)
        return PlottingMethods.to_html(fig)


print('PlottingMethods class defined.')

PlottingMethods class defined.


In [22]:
# Bar Chart
bar_html = PlottingMethods.bar_chart(
    inspector.df, 'Embarked', title='Embarkation Port Distribution'
)
display(HTML(bar_html))

In [23]:
# Pie / Donut Chart
pie_html = PlottingMethods.pie_chart(
    inspector.df, 'Sex', title='Gender Distribution'
)
display(HTML(pie_html))

In [24]:
# Histogram
hist_html = PlottingMethods.histogram(
    inspector.df, 'Age', bins=25, title='Age Distribution'
)
display(HTML(hist_html))